# Production reactions

:::{autolink-concat}
:::

In addition to decay processes with a single initial state, qrules can generate transitions for **production reactions with two initial states** (see [ComPWA/qrules#29](https://github.com/ComPWA/qrules/issues/29)) — think of photoproduction at GlueX ($\gamma p$), fixed-target proton scattering at HADES ($pp$), antiproton annihilation at PANDA ($p\bar p$), or $e^+e^-$ annihilation at BESIII. This page shows how such reactions are built up from topologies, how the resulting transitions are classified into **Mandelstam channels** ($s$, $t$, and $u$), and how to visualize each channel as a single collapsed diagram.

In [ ]:
import graphviz

import qrules
import qrules.io
from qrules.quantum_numbers import EdgeQuantumNumbers
from qrules.topology import create_isobar_topologies, determine_reaction_channel
from qrules.workflow import generate_qn_transitions

PDG = qrules.load_pdg()

## Topologies with two initial states

For more than one initial state, {func}`.create_isobar_topologies` builds the topologies from $2 \to 1$ *production* nodes in addition to the usual $1 \to 2$ *decay* nodes. For a $2 \to 2$ reaction, this results in two shapes: one where the initial states annihilate into a single intermediate edge and one where they are connected through an exchange edge.

In [ ]:
topologies = create_isobar_topologies(
    number_of_final_states=2,
    number_of_initial_states=2,
)
graphviz.Source(qrules.io.asdot(topologies))

The number of topologies grows quickly with the number of final states, because the exchange edge can attach at several places in the decay chain:

In [ ]:
{
    n_final: len(create_isobar_topologies(n_final, number_of_initial_states=2))
    for n_final in (2, 3, 4, 5)
}

## Mandelstam channels

Which Mandelstam variable an intermediate edge carries follows from cutting the topology at that edge. If one side of the cut contains *both* initial states, the edge carries the invariant mass of an $s$-type channel. Otherwise the edge is an *exchange* between the initial states: following the convention $t = (p_1 - p_3)^2$ and $u = (p_1 - p_4)^2$ for $1\,2 \to 3\,4$, it is a $t$-channel if the first initial state and the first final state lie on the same side of the cut and a $u$-channel otherwise. {func}`.determine_mandelstam_channel` implements this classification per edge and {func}`.determine_reaction_channel` summarizes a whole topology:

In [ ]:
[determine_reaction_channel(topology) for topology in topologies]

As an example, take $\pi^0$ photoproduction, $\gamma p \to \pi^0 p$ (GlueX). Note how the ordering of the final state matches the Mandelstam convention: the photon pairs with the $\pi^0$ in the $t$-channel and with the recoil proton in the $u$-channel. The reaction is generated just like a decay — here at the quantum-number level with {func}`.generate_qn_transitions` — and {meth}`.QNReactionInfo.group_by_channel` classifies the transitions:

In [ ]:
%%time
reaction = generate_qn_transitions(
    initial_state=["gamma", "p"],
    final_state=["pi0", "p"],
    particle_db=PDG,
    allowed_intermediate_particles=[
        "Delta(1232)",
        "N(1440)",
        "rho(770)",
        "omega(782)",
    ],
    allowed_interaction_types=["strong", "em"],
)
{
    channel: len(transitions)
    for channel, transitions in reaction.group_by_channel().items()
}

Combined with {func}`.asdot`'s :code:`collapse_graphs` argument, each channel reduces to a few minimal diagrams that together still represent the full reaction object. The intermediate edges list the allowed exchanged or resonant states in $I^G(J^{PC})$ notation. The $s$-channel contains the baryonic $\Delta$ and $N^*$ resonances:

In [ ]:
channels = reaction.group_by_channel()
graphviz.Source(qrules.io.asdot(channels["s"], collapse_graphs=True))

the $t$-channel proceeds through vector and axial-vector meson exchange ($\omega/\rho$-like and $h_1/b_1$-like quantum numbers):

In [ ]:
graphviz.Source(qrules.io.asdot(channels["t"], collapse_graphs=True))

and the $u$-channel exchanges a baryon between the two vertices — the two diagrams are the two orientations of the exchange edge, corresponding to baryon and antibaryon quantum numbers:

In [ ]:
graphviz.Source(qrules.io.asdot(channels["u"], collapse_graphs=True))

## Selecting channels

If only certain channels are of interest, pass :code:`allowed_channels` to restrict the problem sets *before* they are solved. This speeds up the generation, since fewer constraint problems have to be solved:

In [ ]:
%%time
t_channel_reaction = generate_qn_transitions(
    initial_state=["gamma", "p"],
    final_state=["pi0", "p"],
    particle_db=PDG,
    allowed_intermediate_particles=[
        "Delta(1232)",
        "N(1440)",
        "rho(770)",
        "omega(782)",
    ],
    allowed_interaction_types=["strong", "em"],
    allowed_channels="t",
)
len(t_channel_reaction.transitions)

## Particle-level transitions

Production reactions also work through the classic {func}`.generate_transitions` interface, which matches all intermediate edges — including the exchange edges — to particles and generates the spin projections required for a helicity amplitude model. {meth}`.ReactionInfo.group_by_channel` shows which resonances and exchange particles appear in each channel:

In [ ]:
%%time
particle_reaction = qrules.generate_transitions(
    initial_state=["gamma", "p"],
    final_state=["pi0", "p"],
    allowed_intermediate_particles=[
        "Delta(1232)",
        "N(1440)",
        "rho(770)",
        "omega(782)",
    ],
    allowed_interaction_types=["strong", "em"],
    formalism="helicity",
    particle_db=PDG,
)
{
    channel: sorted({
        state.particle.name
        for transition in transitions
        for state in transition.intermediate_states.values()
    })
    for channel, transitions in particle_reaction.group_by_channel().items()
}

Collapsing the full particle-level reaction now labels the intermediate edges by particle name:

In [ ]:
graphviz.Source(qrules.io.asdot(particle_reaction, collapse_graphs=True))

## Reactions at different energies

When there is more than one initial state, the {class}`.MassConservation` rule is switched off, so the collision energy does not enter the problem. Model a specific energy by restricting :code:`allowed_intermediate_particles` to the resonances that are accessible at that energy. For instance, $e^+e^- \to p\bar p\eta$ at BESIII charmonium energies, where the annihilation proceeds through a $J^{PC}=1^{--}$ state and the $p\eta$ subsystem can form $N^*$ resonances:

In [ ]:
%%time
reaction_ee = generate_qn_transitions(
    initial_state=["e+", "e-"],
    final_state=["p", "p~", "eta"],
    particle_db=PDG,
    allowed_intermediate_particles=["J/psi(1S)", "psi(2S)", "N(1535)"],
    allowed_interaction_types=["strong", "em"],
)
len(reaction_ee.transitions)

In [ ]:
graphviz.Source(qrules.io.asdot(reaction_ee, collapse_graphs=True))

Only the $s$-channel survives here: a $t$- or $u$-channel exchange would require an intermediate state that carries lepton number. The spins, parities, and baryon numbers of the intermediate edges confirm the photon-like annihilation state and the nucleon resonances:

In [ ]:
sorted(reaction_ee.group_by_channel())

In [ ]:
{
    (
        state[EdgeQuantumNumbers.spin_magnitude],
        state[EdgeQuantumNumbers.parity],
        state[EdgeQuantumNumbers.baryon_number],
    )
    for transition in reaction_ee.transitions
    for state in transition.intermediate_states.values()
}

:::{seealso}
{doc}`/usage/qn-transitions` for solving reactions at the quantum-number level — without spin projections — which keeps many-body final states tractable, and {doc}`/usage/reaction` for the general workflow.
:::